# 02_DB_Completion

## Objectif

L'objectif de ce notebook est de **compléter les tables `processed.app_train_processed` et `processed.app_test_processed`** en y ajoutant des informations agrégées provenant des autres tables du projet Home Credit déjà chargées en base PostgreSQL.

À ce stade du projet, les tables `processed` contiennent déjà une sélection de variables issues des fichiers `application_train` et `application_test`, avec un premier nettoyage réalisé côté notebook. La suite logique consiste à enrichir ces jeux avec l'information historique disponible dans les autres tables relationnelles :

- `bureau`
- `previous_application`
- `installments_payments`
- `credit_card_balance`
- `pos_cash_balance`

L'enjeu est de **rester à la maille client `SK_ID_CURR`** afin de produire un train et un test enrichis, prêts pour la phase de preprocessing puis de modélisation.


## Démarche

La démarche suivie dans ce notebook est la suivante :

1. se connecter à PostgreSQL en réutilisant la configuration `.env` du repo ;
2. vérifier l'existence et les volumes des tables `raw` et `processed` utiles ;
3. créer des vues d'agrégation par `SK_ID_CURR` sur les tables historiques ;
4. assembler ces agrégats avec `processed.app_train_processed` et `processed.app_test_processed` ;
5. créer deux nouvelles tables finales :
   - `processed.app_train_enriched`
   - `processed.app_test_enriched`
6. contrôler la cohérence des volumes et la présence des nouvelles colonnes.

Le notebook est conçu pour être **rejouable** : les vues sont recréées avec `CREATE OR REPLACE VIEW`, et les tables enrichies sont recréées après `DROP TABLE IF EXISTS`.


In [1]:
from pathlib import Path
import os

import pandas as pd
import psycopg2
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "scripts").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env")

conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=int(os.getenv("PGPORT", 5432)),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    dbname=os.getenv("PGDATABASE", "home_credit"),
)

print("Connexion PostgreSQL ouverte.")
print("Projet :", project_root)


Connexion PostgreSQL ouverte.
Projet : c:\Users\kevin\Documents\P6\P6_MLOps_1-2


## Vérification des tables sources

On commence par contrôler que les tables nécessaires sont bien présentes en base et que leurs volumes sont cohérents. Cela permet de valider les prérequis avant de lancer les agrégations.


In [2]:
tables_check_query = """
SELECT 'application_train' AS table_name, COUNT(*) AS n_rows FROM application_train
UNION ALL
SELECT 'application_test' AS table_name, COUNT(*) AS n_rows FROM application_test
UNION ALL
SELECT 'bureau' AS table_name, COUNT(*) AS n_rows FROM bureau
UNION ALL
SELECT 'previous_application' AS table_name, COUNT(*) AS n_rows FROM previous_application
UNION ALL
SELECT 'installments_payments' AS table_name, COUNT(*) AS n_rows FROM installments_payments
UNION ALL
SELECT 'credit_card_balance' AS table_name, COUNT(*) AS n_rows FROM credit_card_balance
UNION ALL
SELECT 'pos_cash_balance' AS table_name, COUNT(*) AS n_rows FROM pos_cash_balance
UNION ALL
SELECT 'processed.app_train_processed' AS table_name, COUNT(*) AS n_rows FROM processed.app_train_processed
UNION ALL
SELECT 'processed.app_test_processed' AS table_name, COUNT(*) AS n_rows FROM processed.app_test_processed
"""

tables_status = pd.read_sql(tables_check_query, conn)
display(tables_status)

processed_columns_query = """
SELECT table_name, column_name
FROM information_schema.columns
WHERE table_schema = 'processed'
  AND table_name IN ('app_train_processed', 'app_test_processed')
ORDER BY table_name, ordinal_position
"""

processed_columns = pd.read_sql(processed_columns_query, conn)
display(processed_columns.head(20))

for table_name in ["app_train_processed", "app_test_processed"]:
    table_cols = (
        processed_columns.loc[processed_columns["table_name"] == table_name, "column_name"]
        .astype(str)
        .str.upper()
        .tolist()
    )
    if "SK_ID_CURR" not in table_cols:
        raise ValueError(
            f"La colonne SK_ID_CURR est absente de processed.{table_name}. "
            "Il faut rejouer le notebook 01_EDA.ipynb après avoir conservé SK_ID_CURR dans selected_features."
        )

print("SK_ID_CURR est bien présent dans les tables processed utilisées pour les jointures.")


C:\Users\kevin\AppData\Local\Temp\ipykernel_15032\2085624284.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tables_status = pd.read_sql(tables_check_query, conn)


,table_name,n_rows
0,application_test,48744
1,processed.app_test_processed,48744
2,processed.app_train_processed,251289
3,application_train,307511
4,previous_application,1670214
5,bureau,1716428
6,installments_payments,13605401
7,credit_card_balance,3840312
8,pos_cash_balance,10001358


C:\Users\kevin\AppData\Local\Temp\ipykernel_15032\2085624284.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  processed_columns = pd.read_sql(processed_columns_query, conn)


,table_name,column_name
0,app_test_processed,AMT_CREDIT
1,app_test_processed,AMT_GOODS_PRICE
2,app_test_processed,APARTMENTS_AVG
3,app_test_processed,APARTMENTS_MEDI
4,app_test_processed,APARTMENTS_MODE
5,app_test_processed,BASEMENTAREA_AVG
6,app_test_processed,BASEMENTAREA_MEDI
7,app_test_processed,CODE_GENDER
8,app_test_processed,DAYS_BIRTH
9,app_test_processed,DAYS_EMPLOYED


SK_ID_CURR est bien présent dans les tables processed utilisées pour les jointures.


## Fonction utilitaire d'exécution SQL

On définit une petite fonction pour exécuter proprement des blocs SQL de création de vues ou de tables depuis le notebook.


In [3]:
def run_sql(conn, query: str) -> None:
    with conn.cursor() as cur:
        cur.execute(query)
    conn.commit()


## Agrégations par client

Le principe clé est de **ne jamais joindre directement les tables historiques au train ou au test** sans agrégation préalable. 

En effet, les tables comme `bureau`, `previous_application`, `credit_card_balance`, `pos_cash_balance` ou `installments_payments` contiennent plusieurs lignes par client. Une jointure directe dupliquerait les lignes du train et du test.

On crée donc des vues à la maille `SK_ID_CURR`.


In [4]:
bureau_agg_sql = """
CREATE OR REPLACE VIEW processed.bureau_agg AS
SELECT
    "SK_ID_CURR",
    COUNT(*) AS bureau_nb_records,
    AVG("DAYS_CREDIT") AS bureau_avg_days_credit,
    MAX("DAYS_CREDIT") AS bureau_max_days_credit,
    AVG("AMT_CREDIT_SUM") AS bureau_avg_amt_credit_sum,
    AVG("AMT_CREDIT_SUM_DEBT") AS bureau_avg_amt_credit_sum_debt,
    AVG("AMT_CREDIT_SUM_OVERDUE") AS bureau_avg_amt_credit_sum_overdue,
    SUM(CASE WHEN "CREDIT_ACTIVE" = 'Active' THEN 1 ELSE 0 END) AS bureau_nb_active,
    SUM(CASE WHEN "CREDIT_ACTIVE" = 'Closed' THEN 1 ELSE 0 END) AS bureau_nb_closed
FROM bureau
GROUP BY "SK_ID_CURR"
"""

run_sql(conn, bureau_agg_sql)
print("Vue processed.bureau_agg créée.")


Vue processed.bureau_agg créée.


In [5]:
previous_application_agg_sql = """
CREATE OR REPLACE VIEW processed.previous_application_agg AS
SELECT
    "SK_ID_CURR",
    COUNT(*) AS prev_nb_records,
    AVG("AMT_APPLICATION") AS prev_avg_amt_application,
    AVG("AMT_CREDIT") AS prev_avg_amt_credit,
    AVG("AMT_ANNUITY") AS prev_avg_amt_annuity,
    AVG("RATE_DOWN_PAYMENT") AS prev_avg_rate_down_payment,
    SUM(CASE WHEN "NAME_CONTRACT_STATUS" = 'Approved' THEN 1 ELSE 0 END) AS prev_nb_approved,
    SUM(CASE WHEN "NAME_CONTRACT_STATUS" = 'Refused' THEN 1 ELSE 0 END) AS prev_nb_refused
FROM previous_application
GROUP BY "SK_ID_CURR"
"""

run_sql(conn, previous_application_agg_sql)
print("Vue processed.previous_application_agg créée.")


Vue processed.previous_application_agg créée.


In [6]:
installments_agg_sql = """
CREATE OR REPLACE VIEW processed.installments_agg AS
SELECT
    "SK_ID_CURR",
    COUNT(*) AS inst_nb_records,
    AVG("AMT_INSTALMENT") AS inst_avg_amt_instalment,
    AVG("AMT_PAYMENT") AS inst_avg_amt_payment,
    AVG("DAYS_ENTRY_PAYMENT" - "DAYS_INSTALMENT") AS inst_avg_payment_delay,
    MAX("DAYS_ENTRY_PAYMENT" - "DAYS_INSTALMENT") AS inst_max_payment_delay
FROM installments_payments
GROUP BY "SK_ID_CURR"
"""

run_sql(conn, installments_agg_sql)
print("Vue processed.installments_agg créée.")


Vue processed.installments_agg créée.


In [7]:
credit_card_agg_sql = """
CREATE OR REPLACE VIEW processed.credit_card_agg AS
SELECT
    "SK_ID_CURR",
    COUNT(*) AS cc_nb_records,
    AVG("AMT_BALANCE") AS cc_avg_amt_balance,
    AVG("AMT_CREDIT_LIMIT_ACTUAL") AS cc_avg_credit_limit,
    AVG("AMT_DRAWINGS_CURRENT") AS cc_avg_drawings_current,
    AVG("SK_DPD") AS cc_avg_dpd,
    MAX("SK_DPD") AS cc_max_dpd
FROM credit_card_balance
GROUP BY "SK_ID_CURR"
"""

run_sql(conn, credit_card_agg_sql)
print("Vue processed.credit_card_agg créée.")


Vue processed.credit_card_agg créée.


In [8]:
pos_cash_agg_sql = """
CREATE OR REPLACE VIEW processed.pos_cash_agg AS
SELECT
    "SK_ID_CURR",
    COUNT(*) AS pos_nb_records,
    AVG("SK_DPD") AS pos_avg_dpd,
    MAX("SK_DPD") AS pos_max_dpd,
    AVG("CNT_INSTALMENT") AS pos_avg_cnt_instalment,
    AVG("CNT_INSTALMENT_FUTURE") AS pos_avg_cnt_instalment_future
FROM pos_cash_balance
GROUP BY "SK_ID_CURR"
"""

run_sql(conn, pos_cash_agg_sql)
print("Vue processed.pos_cash_agg créée.")


Vue processed.pos_cash_agg créée.


## Contrôle rapide des vues créées

On vérifie ici que les vues d'agrégation produisent bien une ligne par client et qu'elles sont exploitables pour la jointure finale.


In [9]:
views_check_query = """
SELECT 'processed.bureau_agg' AS view_name, COUNT(*) AS n_rows FROM processed.bureau_agg
UNION ALL
SELECT 'processed.previous_application_agg' AS view_name, COUNT(*) AS n_rows FROM processed.previous_application_agg
UNION ALL
SELECT 'processed.installments_agg' AS view_name, COUNT(*) AS n_rows FROM processed.installments_agg
UNION ALL
SELECT 'processed.credit_card_agg' AS view_name, COUNT(*) AS n_rows FROM processed.credit_card_agg
UNION ALL
SELECT 'processed.pos_cash_agg' AS view_name, COUNT(*) AS n_rows FROM processed.pos_cash_agg
"""

pd.read_sql(views_check_query, conn)


C:\Users\kevin\AppData\Local\Temp\ipykernel_15032\2169398200.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(views_check_query, conn)


,view_name,n_rows
0,processed.previous_application_agg,338857
1,processed.bureau_agg,305811
2,processed.credit_card_agg,103558
3,processed.pos_cash_agg,337252
4,processed.installments_agg,339587


## Création des tables enrichies

On réalise maintenant les jointures finales entre :

- les tables de base `processed.app_train_processed` / `processed.app_test_processed`
- les vues d'agrégation précédemment créées

Les jointures sont faites en `LEFT JOIN` sur `SK_ID_CURR` pour conserver l'intégralité du train et du test.


In [10]:
train_enriched_sql = """
DROP TABLE IF EXISTS processed.app_train_enriched;

CREATE TABLE processed.app_train_enriched AS
SELECT
    t.*,
    b.bureau_nb_records,
    b.bureau_avg_days_credit,
    b.bureau_max_days_credit,
    b.bureau_avg_amt_credit_sum,
    b.bureau_avg_amt_credit_sum_debt,
    b.bureau_avg_amt_credit_sum_overdue,
    b.bureau_nb_active,
    b.bureau_nb_closed,
    p.prev_nb_records,
    p.prev_avg_amt_application,
    p.prev_avg_amt_credit,
    p.prev_avg_amt_annuity,
    p.prev_avg_rate_down_payment,
    p.prev_nb_approved,
    p.prev_nb_refused,
    i.inst_nb_records,
    i.inst_avg_amt_instalment,
    i.inst_avg_amt_payment,
    i.inst_avg_payment_delay,
    i.inst_max_payment_delay,
    cc.cc_nb_records,
    cc.cc_avg_amt_balance,
    cc.cc_avg_credit_limit,
    cc.cc_avg_drawings_current,
    cc.cc_avg_dpd,
    cc.cc_max_dpd,
    pos.pos_nb_records,
    pos.pos_avg_dpd,
    pos.pos_max_dpd,
    pos.pos_avg_cnt_instalment,
    pos.pos_avg_cnt_instalment_future
FROM processed.app_train_processed t
LEFT JOIN processed.bureau_agg b
    ON t."SK_ID_CURR" = b."SK_ID_CURR"
LEFT JOIN processed.previous_application_agg p
    ON t."SK_ID_CURR" = p."SK_ID_CURR"
LEFT JOIN processed.installments_agg i
    ON t."SK_ID_CURR" = i."SK_ID_CURR"
LEFT JOIN processed.credit_card_agg cc
    ON t."SK_ID_CURR" = cc."SK_ID_CURR"
LEFT JOIN processed.pos_cash_agg pos
    ON t."SK_ID_CURR" = pos."SK_ID_CURR"
"""

run_sql(conn, train_enriched_sql)
print("Table processed.app_train_enriched créée.")


Table processed.app_train_enriched créée.


In [11]:
test_enriched_sql = """
DROP TABLE IF EXISTS processed.app_test_enriched;

CREATE TABLE processed.app_test_enriched AS
SELECT
    t.*,
    b.bureau_nb_records,
    b.bureau_avg_days_credit,
    b.bureau_max_days_credit,
    b.bureau_avg_amt_credit_sum,
    b.bureau_avg_amt_credit_sum_debt,
    b.bureau_avg_amt_credit_sum_overdue,
    b.bureau_nb_active,
    b.bureau_nb_closed,
    p.prev_nb_records,
    p.prev_avg_amt_application,
    p.prev_avg_amt_credit,
    p.prev_avg_amt_annuity,
    p.prev_avg_rate_down_payment,
    p.prev_nb_approved,
    p.prev_nb_refused,
    i.inst_nb_records,
    i.inst_avg_amt_instalment,
    i.inst_avg_amt_payment,
    i.inst_avg_payment_delay,
    i.inst_max_payment_delay,
    cc.cc_nb_records,
    cc.cc_avg_amt_balance,
    cc.cc_avg_credit_limit,
    cc.cc_avg_drawings_current,
    cc.cc_avg_dpd,
    cc.cc_max_dpd,
    pos.pos_nb_records,
    pos.pos_avg_dpd,
    pos.pos_max_dpd,
    pos.pos_avg_cnt_instalment,
    pos.pos_avg_cnt_instalment_future
FROM processed.app_test_processed t
LEFT JOIN processed.bureau_agg b
    ON t."SK_ID_CURR" = b."SK_ID_CURR"
LEFT JOIN processed.previous_application_agg p
    ON t."SK_ID_CURR" = p."SK_ID_CURR"
LEFT JOIN processed.installments_agg i
    ON t."SK_ID_CURR" = i."SK_ID_CURR"
LEFT JOIN processed.credit_card_agg cc
    ON t."SK_ID_CURR" = cc."SK_ID_CURR"
LEFT JOIN processed.pos_cash_agg pos
    ON t."SK_ID_CURR" = pos."SK_ID_CURR"
"""

run_sql(conn, test_enriched_sql)
print("Table processed.app_test_enriched créée.")


Table processed.app_test_enriched créée.


## Vérifications finales

On contrôle ici que les tables enrichies ont bien conservé le bon nombre de lignes, ce qui valide l'absence de duplication due aux jointures.


In [12]:
final_check_query = """
SELECT 'processed.app_train_processed' AS table_name, COUNT(*) AS n_rows
FROM processed.app_train_processed
UNION ALL
SELECT 'processed.app_test_processed' AS table_name, COUNT(*) AS n_rows
FROM processed.app_test_processed
UNION ALL
SELECT 'processed.app_train_enriched' AS table_name, COUNT(*) AS n_rows
FROM processed.app_train_enriched
UNION ALL
SELECT 'processed.app_test_enriched' AS table_name, COUNT(*) AS n_rows
FROM processed.app_test_enriched
"""

pd.read_sql(final_check_query, conn)


C:\Users\kevin\AppData\Local\Temp\ipykernel_15032\2308563542.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(final_check_query, conn)


,table_name,n_rows
0,processed.app_test_processed,48744
1,processed.app_test_enriched,48744
2,processed.app_train_processed,251289
3,processed.app_train_enriched,251289


In [13]:
sample_query = """
SELECT *
FROM processed.app_train_enriched
LIMIT 5
"""

pd.read_sql(sample_query, conn)


C:\Users\kevin\AppData\Local\Temp\ipykernel_15032\2656336165.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(sample_query, conn)


,AMT_CREDIT,AMT_GOODS_PRICE,APARTMENTS_AVG,APARTMENTS_MEDI,APARTMENTS_MODE,BASEMENTAREA_AVG,BASEMENTAREA_MEDI,CODE_GENDER,DAYS_BIRTH,DAYS_EMPLOYED,...,cc_avg_amt_balance,cc_avg_credit_limit,cc_avg_drawings_current,cc_avg_dpd,cc_max_dpd,pos_nb_records,pos_avg_dpd,pos_max_dpd,pos_avg_cnt_instalment,pos_avg_cnt_instalment_future
0,1293502.5,1129500.0,0.0959,0.0968,0.0924,0.0529,0.0529,F,-16765,-1188,...,NaN,NaN,NaN,NaN,NaN,28,0.000000,0,10.107143,5.785714
1,135000.0,135000.0,NaN,NaN,NaN,NaN,NaN,M,-19046,-225,...,NaN,NaN,NaN,NaN,NaN,4,0.000000,0,3.750000,2.250000
2,312682.5,297000.0,NaN,NaN,NaN,NaN,NaN,F,-19005,-3039,...,0.0,270000.0,0.0,0.0,0.0,21,0.000000,0,12.000000,8.650000
3,513000.0,513000.0,NaN,NaN,NaN,NaN,NaN,M,-19932,-3038,...,NaN,NaN,NaN,NaN,NaN,66,0.000000,0,15.333333,8.969697
4,490495.5,454500.0,NaN,NaN,NaN,NaN,NaN,M,-16941,-1588,...,NaN,NaN,NaN,NaN,NaN,83,339.060241,1294,11.518072,4.108434


## Conclusion

À l'issue de ce notebook, on dispose de deux nouvelles tables enrichies en base :

- `processed.app_train_enriched`
- `processed.app_test_enriched`

Ces tables constituent la base de travail pour la prochaine étape : le preprocessing final orienté modélisation.

La suite logique sera de :

1. charger ces tables enrichies dans un notebook dédié ;
2. gérer les valeurs manquantes ;
3. encoder les variables catégorielles ;
4. standardiser si nécessaire selon les modèles ;
5. entraîner puis comparer plusieurs modèles de classification.
